In [1]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [2]:
sys.path.append(str(Path.cwd().parent))

ROOT = Path(os.getcwd()).resolve().parents[0]
DATA = ROOT / "data"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", test_df.shape)

Train set: (1708, 12)
Test set: (190, 12)


## Training Configuration
Same settings as `base.ipynb`, with a fixed seed so the run can be repeated. The embedding size is read from the data, so Sentence-BERT doesn't need to be loaded.

In [3]:
import torch
from dataclasses import dataclass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Fixed seed so the run can be repeated
torch.manual_seed(0)

embed_dim = len(train_df["question_bert"].iloc[0])

@dataclass
class Config:
    # Training config
    epochs: int         = 50

    # Data shape
    question_dim: int   = embed_dim
    bert_dim: int       = embed_dim

    # Latent sizes
    style_dim: int      = embed_dim
    content_dim: int    = embed_dim
    context_dim: int    = embed_dim

    disc_dim: int       = embed_dim
    gene_dim: int       = embed_dim

    # Optimisation
    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

cfg = Config(epochs=50)

Using device: cuda


In [4]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False) 

C:\Users\bisha\AppData\Local\Temp\ipykernel_63248\4026125994.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)


## Load Disentangler

In [5]:
from models.disentangler import SBERTDisentangler
from models.discriminator import SBERTDiscriminator
from models.generator import SBERTGenerator
from models.encoders import (
    StyleEncoder,
    ContentEncoder,
    ContextEncoder
)
from train.step import *
from experiment.eval import *

In [6]:
model = SBERTDisentangler(cfg).to(device)
gene_optim = torch.optim.Adam(
    model.generator_parameters(),
    lr=cfg.lr_generator
)
disc_optim = torch.optim.Adam(
    model.discriminator_parameters(),
    lr=cfg.lr_discriminator
)

## Untrained Baseline
The same model before any training. If training doesn't beat this, it isn't helping.

In [7]:
untrained_results = evaluate_disentangler(model, train_loader, test_loader, device)

## Training
The mixed training step from `base.ipynb` (`train_step`).

In [8]:
mix_history = []

for epoch in range(cfg.epochs):
    epoch_losses = {}

    for batch in tqdm(train_loader, total=len(train_loader), desc="Training"):
        batch = {key: value.to(device) for key, value in batch.items()}
        losses = train_step(model, batch, gene_optim, disc_optim)

        for name, value in losses.items():
            epoch_losses[name] = (epoch_losses.get(name, 0.0) + value)

    for name in epoch_losses:
        epoch_losses[name] /= len(train_loader)
        
    mix_history.append(epoch_losses)
    print(
        f"epoch {epoch + 1:3d} | "
        f"generator loss {epoch_losses['generator_loss']:.4f} | "
        f"self loss {epoch_losses['self_loss']:.4f} | "
        f"cross loss {epoch_losses['cross_loss']:.4f} | "
        f"adversarial loss {epoch_losses['adversarial_loss']:.4f} | "
        f"discriminator loss {epoch_losses['discriminator_loss']:.4f}"
    )

Training:   0%|          | 0/54 [00:00<?, ?it/s]

Training:   2%|▏         | 1/54 [00:00<00:16,  3.20it/s]

Training:   6%|▌         | 3/54 [00:00<00:06,  8.24it/s]

Training:  11%|█         | 6/54 [00:00<00:03, 14.43it/s]

Training:  17%|█▋        | 9/54 [00:00<00:02, 18.61it/s]

...

Training:  89%|████████▉ | 48/54 [00:01<00:00, 29.23it/s]

Training:  94%|█████████▍| 51/54 [00:01<00:00, 29.07it/s]

Training: 100%|██████████| 54/54 [00:01<00:00, 28.54it/s]

Training: 100%|██████████| 54/54 [00:01<00:00, 28.39it/s]

epoch  50 | generator loss 1.2190 | self loss 0.3996 | cross loss 0.0868 | adversarial loss 0.7327 | discriminator loss 0.6526


## Evaluation
All scores use the **test set** (190 questions) unless marked otherwise. Probes are fitted on the training set.

Chance levels: human/AI = 0.5, retrieval top-1 = 1/190 ≈ 0.005.

### Original metrics on the test set

In [9]:
device = next(model.parameters()).device

results = evaluate_encoders(
    model=model,
    val_loader=test_loader,
    device=device,
)

print(results)

   Encoder                               Metric   Score
0    Style     Human/AI classification accuracy  0.7842
1  Content    Paired retrieval accuracy (top-1)  0.6053
2  Content             Paired cosine similarity  0.6754
3  Context  Question retrieval accuracy (top-1)  0.6237
4  Context           Question cosine similarity  0.7309


### Train vs. test
The same metrics on the training set, to check for overfitting.

In [10]:
train_results   = evaluate_encoders(model=model, val_loader=train_loader, device=device)
test_results    = evaluate_encoders(model=model, val_loader=test_loader, device=device)

train_results["Split"]  = "Train"
test_results["Split"]   = "Test"

print(pd.concat([train_results, test_results]).pivot_table(
    index=["Encoder", "Metric"],
    columns="Split",
    values="Score",
    sort=False
)[["Train", "Test"]])

Split                                         Train    Test
Encoder Metric                                             
Style   Human/AI classification accuracy     0.8803  0.7842
Content Paired retrieval accuracy (top-1)    0.5588  0.6053
        Paired cosine similarity             0.6635  0.6754
Context Question retrieval accuracy (top-1)  0.3858  0.6237
        Question cosine similarity           0.7249  0.7309


### Full evaluation
The raw SBERT vectors, the untrained model and the trained model, scored the same way.

In [11]:
input_results   = evaluate_input(train_loader, test_loader)
trained_results = evaluate_disentangler(model, train_loader, test_loader, device)

input_results["Model"]      = "Raw input"
untrained_results["Model"]  = "Untrained"
trained_results["Model"]    = "Trained"

all_results = pd.concat(
    [input_results, untrained_results, trained_results],
    ignore_index=True
)

### Probe matrix
Each encoder should score high on its own task and close to chance on the other two:
- **Style**: high human/AI accuracy
- **Content**: high paired retrieval
- **Context**: high question retrieval

In [12]:
probe_metrics = [
    "Human/AI probe accuracy",
    "Paired retrieval accuracy (top-1)",
    "Question probe retrieval (top-1)"
]

print(all_results[all_results["Metric"].isin(probe_metrics)].pivot_table(
    index=["Model", "Encoder"],
    columns="Metric",
    values="Score",
    sort=False
)[probe_metrics])

Metric             Human/AI probe accuracy  Paired retrieval accuracy (top-1)  \
Model     Encoder                                                               
Raw input Input                     0.7842                             0.9789   
Untrained Style                     0.7605                             0.9789   
          Content                   0.7605                             0.9763   
          Context                   0.7658                             0.9737   
Trained   Style                     0.7684                             0.2711   
          Content                   0.7684                             0.6053   
          Context                   0.7737                             0.9368   

Metric             Question probe retrieval (top-1)  
Model     Encoder                                    
Raw input Input                              1.0000  
Untrained Style                              1.0000  
          Content                            0.9974  


### Collapse check
If an encoder collapses, true pairs are no closer than random pairs and only a few dimensions are used.

In [13]:
collapse_metrics = [
    "Paired cosine similarity",
    "Random-pair cosine similarity",
    "Effective dimensions"
]

print(all_results[all_results["Metric"].isin(collapse_metrics)].pivot_table(
    index=["Model", "Encoder"],
    columns="Metric",
    values="Score",
    sort=False
)[collapse_metrics])

Metric             Paired cosine similarity  Random-pair cosine similarity  \
Model     Encoder                                                            
Raw input Input                      0.8673                         0.1938   
Untrained Style                      0.9100                         0.4619   
          Content                    0.9083                         0.4638   
          Context                    0.9108                         0.4577   
Trained   Style                      0.2847                        -0.1609   
          Content                    0.6754                         0.1212   
          Context                    0.9438                         0.3799   

Metric             Effective dimensions  
Model     Encoder                        
Raw input Input                 40.6847  
Untrained Style                 29.2391  
          Content               34.3229  
          Context               32.2516  
Trained   Style                  1.7301  
   

### Swaps and reconstruction
Human style + AI content (and the reverse) should rebuild the right response, carry the style donor's style, and keep the content donor's content.

In [14]:
print(all_results[all_results["Encoder"] == "Generator"].pivot_table(
    index="Metric",
    columns="Model",
    values="Score",
    sort=False
)[["Untrained", "Trained"]])

Model                                           Untrained  Trained
Metric                                                            
Self-reconstruction cosine similarity              0.0000   0.3457
Swap rebuilds the right response (top-1)           0.0053   0.0789
Swap style follows the style donor                 0.5000   0.9658
Swap content follows the content donor (top-1)     0.0053   0.0342


## Observations (seed 0, 2026-10-13)

- **Style information is in every encoder.** A fresh probe tells human from AI at about 0.77 from the style, content *and* context encoders. That is no better than the raw input (0.78). The style information hasn't been moved into the style encoder; it is still everywhere.
- **The content encoder keeps less pair information than doing nothing.** Its paired retrieval is 0.61, against 0.98 for both the raw input and the untrained model. Oddly, the context encoder keeps the most (0.94).
- **The question is in every encoder (probe retrieval ≈ 1.00).** This is expected with `SBERT(question + response)` input: the question text is inside every vector, so the context encoder can't be the only place that holds it. Response-only or difference vectors would make this check meaningful.
- **Training squeezed each encoder into a few directions.** Effective dimensions drop from about 30–40 to 1.7 (style), 4.1 (content) and 6.1 (context). This is partial collapse: pairs are still closer than random pairs, but there is little room left for detail.
- **Swaps change the style label but not much else.** The model's own style classifier follows the style donor 97% of the time, but that classifier was trained on these swaps. The swapped vector rebuilds the right response only 8% of the time, and its content points back to the content donor only 3% of the time. Self-reconstruction cosine similarity is 0.35.
- **Style accuracy drops about 10 points from train to test** (0.88 → 0.78). Retrieval looks *higher* on the test set only because there are 190 candidates instead of 1,708, so retrieval scores from different set sizes can't be compared.

**Caveats:** this is one seed on 190 test questions, so differences of a few points may be noise (roughly ±4 points for accuracy and ±7 for retrieval).